In [ ]:
from data_pipeline import *
import torch_geometric.transforms as T

from training_tools.splitter import StratifiedSplitterWithTestHoldout
from training_tools.task import TaskBinaryNodeClassification
from training_tools.trainer import Trainer
from training_tools.validation import CrossValidator
from training_tools.evaluation import Evaluator


pre_transform = T.Compose([
    MarkCommonNodes(edge_types=('AD','PD')),
    DuplicateCommonNodesAndRelabel(suffix_ad='_AD', suffix_pd='_PD'),
    ReindexConsecutive(),
    MergeRelationsToHomogeneous(merge=['AD','PD'], edge_attr_reduce='sum'),
    FilterSmallConnectedComponents(topk=2)
])

ds = NeuroDegAnc2VecDataset(root='data', pre_transform=pre_transform)
data = ds[0]

In [11]:
print(data)

Data(x=[163, 200], edge_index=[2, 660], y=[163], edge_attr=[660, 1], is_common=[227], was_common_before_dup=[163], string_id=[163])


In [17]:
from models import SAGEResidual1L, MLP, GCN
from models.GCN_broken import GCN as GCN_broken
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
from tqdm import tqdm

def compare_models_over_repeats(
    data,                   # Data
    model_factories,           # [factory_A, factory_B, ...]
    splitter_ctor,             # lambda seed, mode: StratifiedNestedSplitter(...)
    task, trainer, mode="transductive",
    seeds=list(range(10)),
):
    # metriche per modello: dict[name] -> list of per-run test F1
    results = {i: [] for i in range(len(model_factories))}

    with tqdm(total=len(seeds), desc="Training Progress", unit="iteration") as pbar:
        for seed in seeds:
            splitter = splitter_ctor(seed, mode)
            parts = splitter.split(data)

            for idx, factory in enumerate(model_factories):
                cv = CrossValidator(trainer=trainer, model_factory=factory, mode=mode, select_by="f1")
                artifacts, summary = cv.run(task, data, parts)

                evaluator = Evaluator(model_factory=factory, mode=mode)
                test_metrics = evaluator.evaluate(task, data, parts, artifacts["best_state"])
                results[idx].append(test_metrics["f1"])  # metrica primaria
            pbar.update(1)

    return results

model_factories = [
    #lambda : SAGEResidual1L(in_channels=data.x.size(1), hidden_channels=64, out_channels=1, dropout=0),
    lambda : GCN_broken(in_channels=data.x.size(1), hidden_channels=64, out_channels=1, dropout=0),
    lambda : GCN(in_channels=data.x.size(1), hidden_channels=64, out_channels=1, dropout=0)
]

# mode = 'inductive'

splitter_ctor = lambda seed, mode: StratifiedSplitterWithTestHoldout(test_size=0.1, n_splits=5, seed=seed, mode=mode)

task    = TaskBinaryNodeClassification(threshold=0.5, pos_weight=None)
trainer = Trainer(lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10)

results1 = compare_models_over_repeats(data, model_factories, splitter_ctor, task, trainer, mode='inductive')
results2 = compare_models_over_repeats(data, model_factories, splitter_ctor, task, trainer, mode='transductive')


def print_stats(results):
    # confronto A vs B (paired)
    A = np.array(results[0], dtype=float)
    B = np.array(results[1], dtype=float)
    diff = A - B
    meanA, meanB = A.mean(), B.mean()
    mean_diff, std_diff = diff.mean(), diff.std(ddof=1)
    n = len(diff)
    # IC 95% (t-interval)
    from scipy.stats import t
    tcrit = t.ppf(0.975, df=n-1) if n>1 else np.nan
    ci95 = (mean_diff - tcrit * std_diff/np.sqrt(n), mean_diff + tcrit * std_diff/np.sqrt(n)) if n>1 else (np.nan,np.nan)

    tstat, p_t = ttest_rel(A, B)       # test t accoppiato
    try:
        wstat, p_w = wilcoxon(A, B)    # non-parametrico
    except ValueError:
        wstat, p_w = np.nan, np.nan

    print(f"Model A mean F1: {meanA:.3f}, Model B mean F1: {meanB:.3f}")
    print(f"Diff mean (A-B): {mean_diff:.3f}  95% CI {ci95}")
    print(f"Paired t-test p={p_t:.4g} | Wilcoxon p={p_w:.4g}")


Training Progress: 100%|██████████| 10/10 [00:31<00:00,  3.16s/iteration]


In [18]:
print('inductive')
print_stats(results1)
print('transductive')
print_stats(results2)

inductive
Model A mean F1: 0.689, Model B mean F1: 0.711
Diff mean (A-B): -0.022  95% CI (np.float64(-0.14915987794811805), np.float64(0.10581756666463028))
Paired t-test p=0.7095 | Wilcoxon p=0.7695
transductive
Model A mean F1: 0.941, Model B mean F1: 0.942
Diff mean (A-B): -0.001  95% CI (np.float64(-0.03673535294822241), np.float64(0.034684070896940346))
Paired t-test p=0.9496 | Wilcoxon p=1
